In [1]:
import torch
import equiv_dens.utils.base as utils
import equiv_dens.utils.orbitals as orbitals
from equiv_dens.utils.spherical_harmonics import spherical_harmonics
import equiv_dens.utils.spherical_harmonics_deriv as sph_deriv
import numpy as np
from torchviz import make_dot
%load_ext autoreload
%autoreload 2

loading config file None


In [121]:
# Setting up distances, directions and coeffs as variables
coords = torch.randn(1, 3, 3)
coords.requires_grad = True
print('coords', coords)
max_L = 5
center = torch.ones(1, 1, 3) / 10
coords = torch.cat([coords, center], dim=1)
d, u = utils.calculate_distances_and_directions(coords, center=center)
s = spherical_harmonics(max_L, u)
print('s requires grad', [ss.requires_grad for ss in s])
for L in range(len(s)):
    zeros = torch.zeros_like(s[L])
    s[L] = torch.where(torch.isnan(s[L]), zeros, s[L])  # making sure there are no nans to avoid NaNs
print('dist', d)
print('unit', u)
scales = [torch.randn(1, 1, 1, 2) for L in range(max_L + 1)]
widths = [torch.randn(1, 1, 1, 2)**2 for L in range(max_L + 1)]
coeffs = [torch.randn(1, 1, 2 * L + 1, 2) for L in range(max_L + 1)]
print([ss.shape for ss in s])
print('scales', scales)
print('widths', widths)
print('coeffs', coeffs)

coords tensor([[[-0.7355, -0.2298, -1.3352],
         [-0.3289, -1.7641,  0.1001],
         [ 0.0728, -0.3107, -1.4438]]], requires_grad=True)
s requires grad [False, True, True, True, True, True]
dist tensor([[[1.6931],
         [1.9128],
         [1.5977],
         [0.0000]]], grad_fn=<LinalgVectorNormBackward0>)
unit tensor([[[ 4.9350e-01,  1.9478e-01,  8.4765e-01],
         [ 2.2421e-01,  9.7454e-01, -7.2777e-05],
         [ 1.7055e-02,  2.5704e-01,  9.6625e-01],
         [        nan,         nan,         nan]]], grad_fn=<DivBackward0>)
[torch.Size([1, 4, 1]), torch.Size([1, 4, 3]), torch.Size([1, 4, 5]), torch.Size([1, 4, 7]), torch.Size([1, 4, 9]), torch.Size([1, 4, 11])]
scales [tensor([[[[-0.7513]]]]), tensor([[[[1.0124]]]]), tensor([[[[-0.3036]]]]), tensor([[[[0.0815]]]]), tensor([[[[1.8741]]]]), tensor([[[[1.0234]]]])]
widths [tensor([[[[2.3483]]]]), tensor([[[[0.1946]]]]), tensor([[[[0.0027]]]]), tensor([[[[0.4857]]]]), tensor([[[[2.5014]]]]), tensor([[[[0.5081]]]])]
coeffs

In [77]:
# basic gto calculation
gto = 0

for L in range(max_L + 1):
    scale = scales[L]
    width = widths[L]
    coeff = coeffs[L]
    sph = s[L].unsqueeze(-1) * coeff
    print('sph shape', sph.shape)
    # if L == 1:
        # print('sL shape', s[L].shape)
        # print('sph grad', torch.autograd.grad(sph[0, 0, 0, 0], coords))
    rbf = orbitals.gaussian_rbf(d.unsqueeze(-1), width, scale, L)
    gto += torch.sum(rbf * sph, dim=(-2, -1))
print(gto)

sph shape torch.Size([1, 4, 1, 2])
sph shape torch.Size([1, 4, 3, 2])
sph shape torch.Size([1, 4, 5, 2])
sph shape torch.Size([1, 4, 7, 2])
sph shape torch.Size([1, 4, 9, 2])
sph shape torch.Size([1, 4, 11, 2])
tensor([[44.7482, -0.6072,  0.1897, -0.1815]], grad_fn=<AddBackward0>)


In [ ]:
# testing autograd calculation for gto
grad = 0
for i in range(3):
    grad += torch.autograd.grad(gto.squeeze()[i], coords, retain_graph=True)[0]
print(grad)

In [124]:
# main cell for calculating GTO gradient, comparing autodiff to explicit gradient
print('coords requires grad', coords.requires_grad)
print('u requires grad', u.requires_grad)
print('s requres grad', s[0].requires_grad)
print('coords', coords)
print('u', u)
gto = 0
gto_grad = 0
s_deriv = sph_deriv.spherical_harmonics_deriv(max_L, u)
# for L in range(len(s_deriv)):
#     zeros = torch.zeros_like(s_deriv[L])
#     s_deriv[L] = torch.where(torch.isnan(s_deriv[L]), zeros, s_deriv[L])  # making sure there are no nans to avoid NaNs
dcoords = torch.eye(3).unsqueeze(0).unsqueeze(0)
print(dcoords.shape)
# TODO: check robustness for zero coords
for L in range(max_L + 1):
    scale = scales[L]
    width = widths[L]
    coeff = coeffs[L]
    sph = s[L].unsqueeze(-1) * coeff
    print('L', L)
    print('s[L] shape', s[L].shape)
    print('s[l] deriv shape', s_deriv[L].shape)
    print('d shape', d.shape)
    print('coords shape', coords.shape)
    print('sph coeff shape', coeff.shape)
    print('width shape', width.shape)
    print('scale shape', scale.shape)
    print('sph shape', sph.shape)
    print('rbf shape', rbf.shape)
    sph_autograd = 0
    du = 0
    if L != 0:
        for i in range(sph.squeeze(0).shape[0]):
            for j in range(sph.squeeze(0).shape[1]):
                for k in range(sph.squeeze(0).shape[2]):
                    sph_autograd += torch.autograd.grad(sph.squeeze(0)[i, j, k], coords, retain_graph=True)[0]
                    du += torch.autograd.grad(sph.squeeze(0)[i, j, k], u, retain_graph=True)[0]
                # print(sph_autograd)
    # print('sph du autograd', du)
    print('s deriv cooeffs size', (s_deriv[L].unsqueeze(-1) * coeff.unsqueeze(-2)).shape)
    sph_grad = (s_deriv[L].unsqueeze(-1) * coeff.unsqueeze(-2)).sum((-1, -3))
    sph_grad_c = sph_grad.unsqueeze(-2) * -(dcoords/d.unsqueeze(-1) -
                 ((coords - center).unsqueeze(-2) *(coords - center).unsqueeze(-1)/d.unsqueeze(-1)**3))
    sph_grad_c = sph_grad_c.sum(-1)
    zeros = torch.zeros_like(sph_grad_c)
    sph_grad_c = torch.where(torch.isnan(sph_grad_c), zeros, sph_grad_c)  # making sure there are no nans to avoid NaNs
    # print('sph_grad_c', sph_grad_c)
    # print('sph_autograd', sph_autograd)
    rbf = orbitals.gaussian_rbf(d.unsqueeze(-1), width, scale, L)
    rbf_autograd = 0
    for i in range(rbf.squeeze(0, -2).shape[0]):
        for j in range(rbf.squeeze(0, -2).shape[1]):
            rbf_autograd += torch.autograd.grad(rbf.squeeze(0, -2)[i, j], coords, retain_graph=True)[0]
    # print('rbf autograd', rbf_autograd)
    print('gaussian rbd grad shape, ', orbitals.gaussian_rbf_deriv(d.unsqueeze(-1), width, scale, L).shape)
    print('coords/d shape', ((coords - center) / d).unsqueeze(-1).shape)
    rbf_grad = orbitals.gaussian_rbf_deriv(d.unsqueeze(-1), width, scale, L) * ((coords - center) / d).unsqueeze(-1)
    print('rbf_grad.shape', rbf_grad.shape)
    zeros = torch.zeros_like(rbf_grad)
    rbf_grad = torch.where(torch.isnan(rbf_grad), zeros, rbf_grad)  # making sure there are no nans to avoid NaNs
    # rbf_grad = rbf_grad.sum(-1)

    # print('rbf_deriv', rbf_grad)
    # print('coords.shape', coords.shape)
    gto_l = torch.sum(rbf * sph, dim=(-2, -1))
    gto_autograd = 0
    for i in range(gto_l.squeeze().shape[0]):
        gto_autograd += torch.autograd.grad(gto_l.squeeze()[i], coords, retain_graph=True)[0]
    # print('rbf shape', rbf.shape)
    # print('rbf grad shape', rbf_grad.shape)
    # print('rbf grad unsqueeze shape', rbf_grad.unsqueeze(-2).shape)
    # print('sph shape', sph.shape)
    # print('sph deriv unsqueeze coords', sph_grad_c.unsqueeze(-1).shape)
    # print('rbf_grad unsqueeze * sph', (rbf_grad.unsqueeze(-2) * sph).shape)
    # print('rbf* sph_grad unsqueeze ', (rbf * sph_grad_c.unsqueeze(-1)).shape)
    print('sph_grad_c', sph_grad_c.shape)
    print('rbf grad', rbf_grad.shape)
    print('rbf grad * sph', (rbf_grad.unsqueeze(-2) * sph.unsqueeze(-3)).shape)
    print('rbf * sph grad', (rbf.unsqueeze(-3) * sph_grad_c.unsqueeze(-1).unsqueeze(-1)).shape)
    gto_deriv = (rbf_grad.unsqueeze(-2) * sph.unsqueeze(-3)).sum((-2))\
                + (rbf.unsqueeze(-3) * sph_grad_c.unsqueeze(-1).unsqueeze(-1)).sum((-2))
    gto_deriv = gto_deriv.sum((-1))
    print('gto_deriv.shape', gto_deriv.shape)
    # print('sph_grad_c', sph_grad_c)
    # print('rbf', rbf)
    # print('sph', sph)
    print('gto autograd', gto_autograd)
    print('gto deriv', gto_deriv)
    gto += gto_l
    gto_grad += gto_deriv

# grad = torch.autograd.grad(gto[0,1], coords, retain_graph=True)
# print(grad)
grad = 0
for i in range(gto.squeeze().shape[0]):
    grad += torch.autograd.grad(gto.squeeze()[i], coords, retain_graph=True)[0]
print('gto', gto)
print('total gto coords grad', grad)
print('explicit gto coords grad', gto_grad)

coords requires grad True
u requires grad True
s requres grad False
coords tensor([[[-0.7355, -0.2298, -1.3352],
         [-0.3289, -1.7641,  0.1001],
         [ 0.0728, -0.3107, -1.4438],
         [ 0.1000,  0.1000,  0.1000]]], grad_fn=<CatBackward0>)
u tensor([[[ 4.9350e-01,  1.9478e-01,  8.4765e-01],
         [ 2.2421e-01,  9.7454e-01, -7.2777e-05],
         [ 1.7055e-02,  2.5704e-01,  9.6625e-01],
         [        nan,         nan,         nan]]], grad_fn=<DivBackward0>)
torch.Size([1, 1, 3, 3])
L 0
s[L] shape torch.Size([1, 4, 1])
s[l] deriv shape torch.Size([1, 4, 1, 3])
d shape torch.Size([1, 4, 1])
coords shape torch.Size([1, 4, 3])
sph coeff shape torch.Size([1, 1, 1, 1])
width shape torch.Size([1, 1, 1, 1])
scale shape torch.Size([1, 1, 1, 1])
sph shape torch.Size([1, 4, 1, 1])
rbf shape torch.Size([1, 4, 1, 1])
torch.Size([1, 4, 1, 3, 1])
gaussian rbd grad shape,  torch.Size([1, 4, 1, 1])
coords/d shape torch.Size([1, 4, 3, 1])
rbf_grad.shape torch.Size([1, 4, 3, 1])
sph_gr

In [ ]:
# calculating gradient for u, the normalized interatomic direction
du = 0.0
for i in range(u.squeeze().shape[0]):
    for j in range(u.squeeze().shape[1]):
        du += torch.autograd.grad(u.squeeze()[i, j], coords, retain_graph=True)[0]

dcoords = torch.eye(3).unsqueeze(0).unsqueeze(-1)
dcoords = dcoords.expand((-1 ,-1 ,-1, 3))

u_deriv = -(1 / d - (coords * torch.sum(coords, dim=(-1), keepdim=True) / d**3))
print('du', du)
print('u_deriv', u_deriv)

In [ ]:
# combining gradient of degree 1 gto with the derivative of u via chain rule
sph_grad = (s_deriv[1].unsqueeze(-1) * coeffs[1][..., [2, 0, 1], :]).sum(-1)
# sph_grad = (s_deriv[2].unsqueeze(-1) * coeffs[2]).sum((-1, -2))
print('sph grad shape', sph_grad.shape)
dcoords = torch.eye(3).unsqueeze(0).unsqueeze(0)
# dcoords = dcoords.expand((-1 , 3 ,-1, -1))
u_deriv = -(dcoords/d.unsqueeze(-1) - (coords.unsqueeze(-2) * coords.unsqueeze(-1)/d.unsqueeze(-1)**3))
print('u_deriv shape', u_deriv.shape)
print('u_deriv', (sph_grad.unsqueeze(-2) * u_deriv).sum(-1))

In [ ]:
# further tests for calculating the derivative of the degree 1 GTO with plotting of gradient graph
coords2 = torch.tensor(coords)
coords2.requires_grad = True
coeffs1 = torch.tensor(coeffs[1])
class Direction(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.dummy = torch.nn.Parameter(torch.ones((1, 3, 3)))

    def forward(self, Ri):
        Ri = Ri * self.dummy
        Rj = torch.zeros_like(Ri)
        rij = Rj - Ri  # displacement vectors
        dij = torch.norm(rij, dim=-1, keepdim=True)  # distances
        uij = rij / dij  # unit displacement vectors
        return uij

class L1(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.dummy = torch.nn.Parameter(torch.ones((1, 3, 3)))

    def forward(self, u):
        u = u * self.dummy
        print('u', u)
        return np.sqrt(3) * u[..., [1, 2, 0]] 

class SphCoeffs(torch.nn.Module):
    def __init__(self, coeffs):
        super().__init__()
        self.coeffs = torch.nn.Parameter(coeffs)

    def forward(self, s):
        return s.unsqueeze(-1) * self.coeffs

l1 = L1()
dir = Direction()
sphcoeffs = SphCoeffs(coeffs1)
model = torch.nn.Sequential(dir, l1, sphcoeffs)

print('coords', coords2)

sph_l1 = model(coords2)
# make_dot(sph_l1.mean(), params=dict(model.named_parameters()))
print("l1", sph_l1)
l = sph_l1.sum()
l.backward()
print('coords_grad', coords2.grad)
print('l1 dummy', l1.dummy.grad)
print('dir dummy', dir.dummy.grad)

In [ ]:
# calculating gradient graph for degree 1 gto detailed steps version with initial gradien from spherical harmonics
coords2 = coords.clone().detach()
coords2.requires_grad = True
coeffs1 = coeffs[2].clone().detach()
c_pow2 = coords2**2
c_pow2.retain_grad()
c_pow2_sum = c_pow2.sum(-1, keepdim=True)
c_pow2_sum.retain_grad()
d = c_pow2_sum**(0.5)
d.retain_grad()
u2 = coords2 * d**(-1)  # unit displacement vectors
u2.retain_grad()
l = (u2 * sph_grad).sum()
# l = u2.sum()
l.backward(retain_graph=True)
print('u grad', u2.grad)
print('d grad', d.grad)
print('c_pow2_sum grad', c_pow2_sum.grad)
print('c_pow2 grad', c_pow2.grad)
print('one side?', 2 * coords2 * c_pow2.grad)
print('coords grad', coords2.grad)
print('coords', coords)
print('final grad?', 2 * coords2 * c_pow2.grad + sph_grad/d)
make_dot(l)

In [ ]:
# explicitly calculating gradient for degree 1 GTO, ensuring correct dimensions and broadcasting
s_deriv = sph_deriv.spherical_harmonics_deriv(2, u)

sph_grad = (s_deriv[2].unsqueeze(-1) * coeffs[2]).sum((-1,-2))
print('sph_grad.shape', sph_grad.shape)

dudr = -(1 / d - (coords * torch.sum(coords, dim=(-1), keepdim=True) / d**3)) 
dudd = (coords / -d**2)
dddp = 1/(2 *  torch.sqrt(torch.sum(coords**2, -1, keepdim=True)))
dpds = 2*coords
sph_grad_c = sph_grad * dudr
print('s grad', s_deriv[1])
print('sph_grad', sph_grad)
print('u grad r', dudr)
print('u grad d', (sph_grad*dudd).sum(-1, keepdim=True))
print('d grad sqrt', (sph_grad*dudd*dddp).sum(-1, keepdim=True))
print('sqrt grad sum', dpds*(sph_grad*dddp*dudd).sum(-1, keepdim=True))
print('sqrt grad sum', dpds*(sph_grad*dddp*dudd).sum(-1, keepdim=True))
print('sum up', sph_grad*1/d + dpds*(sph_grad*dddp*dudd).sum(-1, keepdim=True))
# print('second part', (coords * torch.sum(coords, dim=(-1), keepdim=True) / d**3))
# print('sph_grad_c', sph_grad_c)